# Notebook2：DAS定位消融与完整评估

本notebook不仅做消融试验，还包括基线运行、噪声鲁棒性、参数敏感性和汇总报表。

## 1. 初始化与参数区

In [ ]:
#  VSCode/Jupyter  src 
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "src").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "src").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.das_localization.io_utils import read_phase_bin_file, infer_points_per_frame
from src.das_localization.localization import LocalizeConfig, localize_single_event

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
print(f"PROJECT_ROOT={PROJECT_ROOT}")

# 
DATA_FILE = r"E:\codes\ZZ-BK\data\eDAS_loc\eDAS-2000Hz-0032pt-20260323T110032.578.bin"
OUTPUT_ROOT = Path(r"E:\codes\ZZ-BK\outputs\das_localization")
(OUTPUT_ROOT / "figures").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "metrics").mkdir(parents=True, exist_ok=True)

# 
FS = 2000.0
DX = 3.2
POINTS = infer_points_per_frame(DATA_FILE)
data = read_phase_bin_file(DATA_FILE, points_per_frame=POINTS)

# 
base_cfg = LocalizeConfig(
    fs=FS, dx=DX, velocity=343.0,
    ref_mode="best_snr", ref_ch=15,
    delay_method="gcc_phat", max_tau_sec=0.02, psr_th=6.0, n_valid_min=8,
    fit_model="hyperbola", fit_solver="least_squares", robust_loss="linear",
    use_ransac=True, ransac_iters=200, ransac_resid_th=8e-4,
    x_bounds=(-20.0, 120.0), z_bounds=(0.1, 5.0),
    do_demean=True, do_highpass=True, hp_cutoff=100.0, hp_order=2,
    do_bandpass=False, bp_lowcut=120.0, bp_highcut=800.0, bp_order=4,
    normalize=True, rmse_th=1e-3, conf_th=0.6,
)

def run_cfg(cfg, x):
    """"""
    r = localize_single_event(x, cfg)
    return {
        "source_x": r.get("source_x"),
        "source_z": r.get("source_z"),
        "rmse_tdoa": r.get("rmse_tdoa"),
        "confidence": r.get("confidence"),
        "status": r.get("status"),
        "reason": r.get("reason"),
        "valid_channels": r.get("valid_channels"),
    }

## 2. 基线运行（检查管道是否正常）

In [ ]:
# 
baseline = run_cfg(base_cfg, data)
baseline_df = pd.DataFrame([baseline])
print(baseline_df)

## 3. 消融试验1：时延估计方式对比

In [ ]:
rows = []
for method in ["ncc", "fft_xcorr", "gcc_phat"]:
    cfg = deepcopy(base_cfg)
    cfg.delay_method = method
    out = run_cfg(cfg, data)
    out["ablation"] = "delay_method"
    out["setting"] = method
    rows.append(out)

df_delay = pd.DataFrame(rows)
print(df_delay)

## 4. 消融试验2：双曲线 vs 抛物线

In [ ]:
rows = []
for model in ["hyperbola", "parabola"]:
    cfg = deepcopy(base_cfg)
    cfg.fit_model = model
    out = run_cfg(cfg, data)
    out["ablation"] = "fit_model"
    out["setting"] = model
    rows.append(out)

df_model = pd.DataFrame(rows)
print(df_model)

## 5. 消融试验3：最小二乘 vs 鲁棒最小二乘

In [ ]:
rows = []
solver_settings = [
    ("least_squares", "linear"),
    ("robust", "huber"),
    ("robust", "soft_l1"),
    ("robust", "cauchy"),
]
for solver, loss in solver_settings:
    cfg = deepcopy(base_cfg)
    cfg.fit_solver = solver
    cfg.robust_loss = loss
    out = run_cfg(cfg, data)
    out["ablation"] = "solver"
    out["setting"] = f"{solver}:{loss}"
    rows.append(out)

df_solver = pd.DataFrame(rows)
print(df_solver)

## 6. 消融试验4：是否启用 RANSAC

In [ ]:
rows = []
for use_ransac in [False, True]:
    cfg = deepcopy(base_cfg)
    cfg.use_ransac = use_ransac
    out = run_cfg(cfg, data)
    out["ablation"] = "ransac"
    out["setting"] = str(use_ransac)
    rows.append(out)

df_ransac = pd.DataFrame(rows)
print(df_ransac)

## 7. 噪声鲁棒性测试（添加高斯噪声）

In [ ]:
def add_noise_by_snr(x, snr_db, rng):
    """SNR"""
    sig_pow = np.mean(x**2)
    noise_pow = sig_pow / (10 ** (snr_db / 10.0))
    noise = rng.normal(0.0, np.sqrt(noise_pow), size=x.shape)
    return x + noise

rng = np.random.default_rng(20260518)
rows = []
for snr_db in [20, 10, 0, -5]:
    noisy = add_noise_by_snr(data, snr_db, rng)
    out = run_cfg(base_cfg, noisy)
    out["ablation"] = "noise"
    out["setting"] = snr_db
    rows.append(out)

df_noise = pd.DataFrame(rows)
print(df_noise)

## 8. 参数敏感性分析

In [ ]:
rows = []

# PSR 
for psr_th in [4.0, 5.0, 6.0, 7.0, 8.0]:
    cfg = deepcopy(base_cfg)
    cfg.psr_th = psr_th
    out = run_cfg(cfg, data)
    out["ablation"] = "psr_th"
    out["setting"] = psr_th
    rows.append(out)

# 
for vel in [320.0, 343.0, 360.0]:
    cfg = deepcopy(base_cfg)
    cfg.velocity = vel
    out = run_cfg(cfg, data)
    out["ablation"] = "velocity"
    out["setting"] = vel
    rows.append(out)

# 
for tau in [0.01, 0.02, 0.03, 0.05]:
    cfg = deepcopy(base_cfg)
    cfg.max_tau_sec = tau
    out = run_cfg(cfg, data)
    out["ablation"] = "max_tau_sec"
    out["setting"] = tau
    rows.append(out)

df_sens = pd.DataFrame(rows)
print(df_sens.head(12))

## 9. 汇总保存与可视化输出

In [ ]:
# 
ablation_main = pd.concat([df_delay, df_model, df_solver, df_ransac], ignore_index=True)
ablation_main.to_csv(OUTPUT_ROOT / "metrics" / "ablation_main.csv", index=False, encoding="utf-8-sig")
df_noise.to_csv(OUTPUT_ROOT / "metrics" / "ablation_noise.csv", index=False, encoding="utf-8-sig")
df_sens.to_csv(OUTPUT_ROOT / "metrics" / "ablation_sensitivity.csv", index=False, encoding="utf-8-sig")
baseline_df.to_csv(OUTPUT_ROOT / "metrics" / "ablation_baseline.csv", index=False, encoding="utf-8-sig")

# 1RMSE
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_delay["setting"].astype(str), df_delay["rmse_tdoa"].astype(float))
ax.set_title("RMSE")
ax.set_xlabel("")
ax.set_ylabel("RMSE")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "figures" / "nb2_delay_method_rmse.png", dpi=160)

# 2
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_noise["setting"].astype(float), df_noise["rmse_tdoa"].astype(float), marker="o")
ax.set_title("SNR vs RMSE")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("RMSE")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "figures" / "nb2_noise_robustness.png", dpi=160)

# 3PSR
psr_rows = df_sens[df_sens["ablation"] == "psr_th"].copy()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(psr_rows["setting"].astype(float), psr_rows["rmse_tdoa"].astype(float), marker="o")
ax.set_title("PSR vs RMSE")
ax.set_xlabel("PSR")
ax.set_ylabel("RMSE")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "figures" / "nb2_psr_sensitivity.png", dpi=160)

print(" outputs/das_localization")